# Benchmark: Folding Severity Sweep

Generates DVFs with increasing levels of folding by scaling displacement
magnitude, and measures how the optimizer copes:

- **Magnitude sweep**: scale a base random field from mild to extreme
- **Density sweep**: increase the number of folding sources (correspondences)

Metrics: initial neg-Jdet count, runtime, final L2 error, convergence success.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

from dvfopt import (
    iterative_parallel,
    jacobian_det2D,
    generate_random_dvf,
    scale_dvf,
)
import laplacian
from benchmark_utils import (
    run_correction,
    plot_jac_heatmaps,
    plot_correction_magnitude,
    plot_jdet_histograms,
)

---
## Part 1: Magnitude Sweep

A fixed 5x5 random pattern is upscaled to 40x40 with varying
`max_magnitude` values.  Higher magnitude → more aggressive
displacements → more folding.

In [ ]:
MAGNITUDES = [1.0, 2.0, 3.0, 5.0, 7.0, 10.0, 15.0, 20.0]
TARGET_SIZE = (40, 40)
SEED = 42

In [ ]:
mag_results = {}

print(f"{'Magnitude':>10s}  {'Neg init':>10s}  {'Neg final':>10s}  "
      f"{'Time (s)':>10s}  {'Min Jdet':>10s}  {'L2 Error':>10s}")
print("-" * 72)

for mag in MAGNITUDES:
    dvf = generate_random_dvf((3, 1, 5, 5), mag, SEED)
    dvf = scale_dvf(dvf, TARGET_SIZE)
    mag_results[mag] = run_correction(dvf, iterative_parallel)
    r = mag_results[mag]
    status = "OK" if r["n_neg_final"] == 0 else "FAIL"
    print(f"{mag:>10.1f}  {r['n_neg_init']:>10d}  {r['n_neg_final']:>10d}  "
          f"{r['time']:>10.2f}  {r['min_jdet']:>+10.6f}  {r['l2_err']:>10.4f}  [{status}]")

In [ ]:
mags = list(mag_results.keys())
m_negs = [mag_results[m]["n_neg_init"] for m in mags]
m_negs_final = [mag_results[m]["n_neg_final"] for m in mags]
m_times = [mag_results[m]["time"] for m in mags]
m_l2s = [mag_results[m]["l2_err"] for m in mags]
m_success = [mag_results[m]["n_neg_final"] == 0 for m in mags]
m_min_init = [mag_results[m]["min_jdet_init"] for m in mags]
m_min_final = [mag_results[m]["min_jdet"] for m in mags]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# Row 1: existing plots
ax = axes[0, 0]
ax.plot(mags, m_negs, "o-", color="tab:red")
ax.set_xlabel("Max displacement magnitude")
ax.set_ylabel("Neg Jdet pixels (initial)")
ax.set_title("Folding vs Magnitude")
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
colors = ["tab:green" if s else "tab:red" for s in m_success]
ax.bar([str(m) for m in mags], m_times, color=colors)
ax.set_xlabel("Max magnitude")
ax.set_ylabel("Time (s)")
ax.set_title("Runtime (green=converged, red=failed)")

ax = axes[0, 2]
ax.plot(m_negs, m_l2s, "s-", color="tab:orange")
ax.set_xlabel("Initial neg-Jdet pixels")
ax.set_ylabel("L2 Error")
ax.set_title("L2 Cost vs Folding Amount")
ax.grid(True, alpha=0.3)

# Row 2: new plots
ax = axes[1, 0]
x = np.arange(len(mags))
w = 0.35
ax.bar(x - w / 2, m_min_init, w, label="Before", color="tab:red", alpha=0.7)
ax.bar(x + w / 2, m_min_final, w, label="After", color="tab:blue", alpha=0.7)
ax.axhline(0, color="black", linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels([str(m) for m in mags], rotation=30)
ax.set_xlabel("Max magnitude")
ax.set_ylabel("Min Jacobian determinant")
ax.set_title("Worst-case Jdet: Before vs After")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis="y")

ax = axes[1, 1]
ax.plot(mags, m_negs, "o--", color="tab:red", label="Before", alpha=0.6)
ax.plot(mags, m_negs_final, "s-", color="tab:blue", label="After", linewidth=2)
ax.set_xlabel("Max magnitude")
ax.set_ylabel("Neg-Jdet pixel count")
ax.set_title("Convergence Success vs Magnitude")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 2]
time_per_neg = [mag_results[m]["time"] / max(mag_results[m]["n_neg_init"], 1)
                for m in mags]
ax.bar([str(m) for m in mags], time_per_neg, color="tab:purple")
ax.set_xlabel("Max magnitude")
ax.set_ylabel("Time per neg-Jdet pixel (s)")
ax.set_title("Correction Cost per Folding Pixel")
ax.grid(True, alpha=0.3, axis="y")

plt.suptitle("Magnitude Sweep — 40x40 grid", fontsize=14)
plt.tight_layout()
plt.show()

### Jacobian heatmaps — before vs after correction

Each column is one magnitude level. Top row = initial Jacobian, bottom row = corrected.
Red regions have negative Jacobian determinant (folding).

In [ ]:
mags = list(mag_results.keys())
plot_jac_heatmaps(
    [[mag_results[m]["jac_init"][0] for m in mags],
     [mag_results[m]["jac_final"][0] for m in mags]],
    [f"mag={m}" for m in mags],
    title="Jacobian Determinant — Before vs After (Magnitude Sweep)",
)
plt.show()

In [ ]:
plot_correction_magnitude(
    [(mag_results[m]["phi"], mag_results[m]["phi_init"]) for m in mags],
    [f"mag={m}" for m in mags],
    title="Per-pixel displacement change (correction magnitude)",
)
plt.show()

In [ ]:
plot_jdet_histograms(
    [[("Before", mag_results[m]["jac_init"][0]),
      ("After", mag_results[m]["jac_final"][0])] for m in mags],
    [f"mag={m}" for m in mags],
    title="Jacobian Determinant Distribution (Magnitude Sweep)",
)
plt.show()

---
## Part 2: Folding Density Sweep

Varies the number of crossing correspondence pairs on a fixed 30x30 grid.
More pairs → more spatially distributed folding regions.

In [ ]:
def make_crossing_pairs(n_pairs, H, W, seed=0):
    """Generate n crossing correspondence pairs spread across the grid."""
    rng = np.random.default_rng(seed)
    margin = 3
    msample = []
    fsample = []

    for _ in range(n_pairs):
        # Two nearby points that swap positions
        y1 = rng.integers(margin, H - margin)
        x1 = rng.integers(margin, W - margin)
        dy = rng.integers(2, 5)
        dx = rng.integers(2, 5)
        y2 = min(y1 + dy, H - margin - 1)
        x2 = min(x1 + dx, W - margin - 1)

        msample.append([0, y1, x1])
        msample.append([0, y2, x2])
        fsample.append([0, y2, x2])
        fsample.append([0, y1, x1])

    return np.array(msample), np.array(fsample)

In [ ]:
PAIR_COUNTS = [1, 2, 4, 6, 8, 12, 16, 20]
GRID_H, GRID_W = 30, 30

density_results = {}

print(f"{'Pairs':>8s}  {'Neg init':>10s}  {'Neg final':>10s}  "
      f"{'Time (s)':>10s}  {'Min Jdet':>10s}  {'L2 Error':>10s}")
print("-" * 68)

for n_pairs in PAIR_COUNTS:
    ms, fs = make_crossing_pairs(n_pairs, GRID_H, GRID_W, seed=n_pairs)
    fixed_sample = np.zeros((1, GRID_H, GRID_W))
    dvf = laplacian.solveLaplacianFromCorrespondences(fixed_sample.shape, ms, fs)
    density_results[n_pairs] = run_correction(dvf, iterative_parallel)
    r = density_results[n_pairs]
    status = "OK" if r["n_neg_final"] == 0 else "FAIL"
    print(f"{n_pairs:>8d}  {r['n_neg_init']:>10d}  {r['n_neg_final']:>10d}  "
          f"{r['time']:>10.2f}  {r['min_jdet']:>+10.6f}  {r['l2_err']:>10.4f}  [{status}]")

In [ ]:
pairs = list(density_results.keys())
d_negs = [density_results[p]["n_neg_init"] for p in pairs]
d_negs_final = [density_results[p]["n_neg_final"] for p in pairs]
d_times = [density_results[p]["time"] for p in pairs]
d_l2s = [density_results[p]["l2_err"] for p in pairs]
d_success = [density_results[p]["n_neg_final"] == 0 for p in pairs]
d_min_init = [density_results[p]["min_jdet_init"] for p in pairs]
d_min_final = [density_results[p]["min_jdet"] for p in pairs]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# Row 1: original plots
ax = axes[0, 0]
ax.plot(pairs, d_negs, "o-", color="tab:red")
ax.set_xlabel("Number of crossing pairs")
ax.set_ylabel("Neg Jdet pixels (initial)")
ax.set_title("Folding vs Pair Count")
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
colors = ["tab:green" if s else "tab:red" for s in d_success]
ax.bar([str(p) for p in pairs], d_times, color=colors)
ax.set_xlabel("Crossing pairs")
ax.set_ylabel("Time (s)")
ax.set_title("Runtime (green=converged, red=failed)")

ax = axes[0, 2]
ax.plot(pairs, d_l2s, "s-", color="tab:orange")
ax.set_xlabel("Number of crossing pairs")
ax.set_ylabel("L2 Error")
ax.set_title("L2 Cost vs Pair Count")
ax.grid(True, alpha=0.3)

# Row 2: new plots
ax = axes[1, 0]
x = np.arange(len(pairs))
w = 0.35
ax.bar(x - w / 2, d_min_init, w, label="Before", color="tab:red", alpha=0.7)
ax.bar(x + w / 2, d_min_final, w, label="After", color="tab:blue", alpha=0.7)
ax.axhline(0, color="black", linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels([str(p) for p in pairs])
ax.set_xlabel("Crossing pairs")
ax.set_ylabel("Min Jacobian determinant")
ax.set_title("Worst-case Jdet: Before vs After")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis="y")

ax = axes[1, 1]
ax.plot(pairs, d_negs, "o--", color="tab:red", label="Before", alpha=0.6)
ax.plot(pairs, d_negs_final, "s-", color="tab:blue", label="After", linewidth=2)
ax.set_xlabel("Crossing pairs")
ax.set_ylabel("Neg-Jdet pixel count")
ax.set_title("Convergence Success vs Density")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 2]
time_per_neg = [density_results[p]["time"] / max(density_results[p]["n_neg_init"], 1)
                for p in pairs]
ax.bar([str(p) for p in pairs], time_per_neg, color="tab:purple")
ax.set_xlabel("Crossing pairs")
ax.set_ylabel("Time per neg-Jdet pixel (s)")
ax.set_title("Correction Cost per Folding Pixel")
ax.grid(True, alpha=0.3, axis="y")

plt.suptitle(f"Density Sweep — {GRID_H}x{GRID_W} grid", fontsize=14)
plt.tight_layout()
plt.show()

### Jacobian heatmaps — before vs after (Density Sweep)

In [ ]:
pairs = list(density_results.keys())
plot_jac_heatmaps(
    [[density_results[p]["jac_init"][0] for p in pairs],
     [density_results[p]["jac_final"][0] for p in pairs]],
    [f"{p} pairs" for p in pairs],
    title="Jacobian Determinant — Before vs After (Density Sweep)",
)
plt.show()

In [ ]:
plot_jdet_histograms(
    [[("Before", density_results[p]["jac_init"][0]),
      ("After", density_results[p]["jac_final"][0])] for p in pairs],
    [f"{p} pairs" for p in pairs],
    title="Jacobian Determinant Distribution (Density Sweep)",
)
plt.show()

---
## Combined: Runtime vs Initial Neg-Jdet Pixels

Overlay both sweeps to see if runtime correlates with the number of
folding pixels regardless of how the folding was produced.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# Runtime vs initial neg-Jdet pixels (colored by outcome)
ax = axes[0, 0]
for i, mag in enumerate(mags):
    r = mag_results[mag]
    c = "tab:green" if r["n_neg_final"] == 0 else "tab:red"
    ax.scatter(r["n_neg_init"], r["time"], marker="o", s=80, color=c, zorder=3,
               edgecolors="black", linewidth=0.5)
for i, p in enumerate(pairs):
    r = density_results[p]
    c = "tab:green" if r["n_neg_final"] == 0 else "tab:red"
    ax.scatter(r["n_neg_init"], r["time"], marker="s", s=80, color=c, zorder=3,
               edgecolors="black", linewidth=0.5)
# Legend: manual
from matplotlib.lines import Line2D
legend_els = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="gray", markersize=8, label="Magnitude sweep"),
    Line2D([0], [0], marker="s", color="w", markerfacecolor="gray", markersize=8, label="Density sweep"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="tab:green", markersize=8, label="Converged"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="tab:red", markersize=8, label="Failed"),
]
ax.legend(handles=legend_els, fontsize=8)
ax.set_xlabel("Initial neg-Jdet pixels")
ax.set_ylabel("Time (s)")
ax.set_title("Runtime vs Folding Severity")
ax.grid(True, alpha=0.3)

# L2 error vs initial neg-Jdet pixels (colored by outcome)
ax = axes[0, 1]
for i, mag in enumerate(mags):
    r = mag_results[mag]
    c = "tab:green" if r["n_neg_final"] == 0 else "tab:red"
    ax.scatter(r["n_neg_init"], r["l2_err"], marker="o", s=80, color=c, zorder=3,
               edgecolors="black", linewidth=0.5)
for i, p in enumerate(pairs):
    r = density_results[p]
    c = "tab:green" if r["n_neg_final"] == 0 else "tab:red"
    ax.scatter(r["n_neg_init"], r["l2_err"], marker="s", s=80, color=c, zorder=3,
               edgecolors="black", linewidth=0.5)
ax.set_xlabel("Initial neg-Jdet pixels")
ax.set_ylabel("L2 Error")
ax.set_title("L2 Deviation vs Folding Severity")
ax.grid(True, alpha=0.3)

# Initial min-Jdet vs runtime (colored by sweep type)
ax = axes[1, 0]
ax.scatter(m_min_init, m_times, marker="o", s=80, label="Magnitude sweep",
           color="tab:blue", zorder=3)
ax.scatter(d_min_init, d_times, marker="s", s=80, label="Density sweep",
           color="tab:orange", zorder=3)
ax.set_xlabel("Initial min Jacobian determinant")
ax.set_ylabel("Time (s)")
ax.set_title("Runtime vs Worst Initial Jdet")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# % pixels fixed vs initial severity
ax = axes[1, 1]
total_pixels_mag = TARGET_SIZE[0] * TARGET_SIZE[1]
total_pixels_den = GRID_H * GRID_W
pct_fixed_mag = [100 * (mag_results[m]["n_neg_init"] - mag_results[m]["n_neg_final"])
                 / max(mag_results[m]["n_neg_init"], 1) for m in mags]
pct_fixed_den = [100 * (density_results[p]["n_neg_init"] - density_results[p]["n_neg_final"])
                 / max(density_results[p]["n_neg_init"], 1) for p in pairs]
pct_init_mag = [100 * mag_results[m]["n_neg_init"] / total_pixels_mag for m in mags]
pct_init_den = [100 * density_results[p]["n_neg_init"] / total_pixels_den for p in pairs]

ax.scatter(pct_init_mag, pct_fixed_mag, marker="o", s=80, label="Magnitude sweep",
           color="tab:blue", zorder=3)
ax.scatter(pct_init_den, pct_fixed_den, marker="s", s=80, label="Density sweep",
           color="tab:orange", zorder=3)
ax.axhline(100, color="gray", linewidth=0.8, linestyle="--", alpha=0.5)
ax.set_xlabel("% grid initially folded")
ax.set_ylabel("% neg-Jdet pixels fixed")
ax.set_title("Correction Success Rate vs Severity")
ax.legend(fontsize=8)
ax.set_ylim(None, 105)
ax.grid(True, alpha=0.3)

plt.suptitle("Combined Severity Analysis", fontsize=14)
plt.tight_layout()
plt.show()

## Summary

Key takeaways:
- **Magnitude**: higher displacement magnitudes create more folding and require more correction time and L2 deviation
- **Density**: more crossing regions increase both the number of folding pixels and total correction cost
- **Convergence**: at extreme severity levels the optimizer may not fully converge within default iteration limits
- **Cost scaling**: the per-pixel correction cost varies with severity — densely folded fields may amortize overhead better
- **Jacobian heatmaps**: visually confirm that all negative regions (red) are eliminated after correction
- **Success rate**: the combined scatter shows what fraction of folding pixels are fixed at each severity level